# 08 Gold — Produtos Analíticos e Métricas Operacionais

Este notebook implementa a camada **Gold** do pipeline de dados eleitorais (`tse-analysis-08-gold-analytics`), atendendo aos requisitos e perguntas analíticas da **Issue #16**:

1. **Distribuição e Frequência de Eventos (`gold_event_distribution`):**
   - Qual a frequência, volume absoluto e proporção percentual de cada categoria/código de evento operacional registrado nas urnas eletrônicas por Turno e Unidade Federativa (UF)?
2. **Taxa de Completude de Sessões de Votação (`gold_session_completeness`):**
   - Qual a taxa de sucesso/encerramento regular (`CLOSED`) versus sessões incompletas ou interrompidas (`INCOMPLETE`) nas urnas eletrônicas?
   - Qual o comportamento operacional em termos de ciclos de votantes registrados (`voter_cycles`) e eventos totais agregados por Turno, UF e estado final da sessão?
3. **Padrões de Tentativas Biométricas (`gold_biometric_attempts`):**
   - Como se distribuem as tentativas biométricas dos eleitores (1ª, 2ª, 3ª, 4ª tentativa)?
   - Qual a proporção e a curva cumulativa de tentativas por Turno e UF, evidenciando padrões de identificação e possíveis pontos de atrito no terminal do mesário?

---
### Arquitetura e Contratos
- **Entradas Silver:** `fact_urna_event` e `fact_voting_session` (lidas de `{DATASET_PATH}/silver`).
- **Saídas Gold:** Gravadas em `{DATASET_PATH}/gold/` em formato Parquet comprimido (ZSTD nível 5).
- **Resiliência e Fallbacks:** Leitura defensiva com criação automática de dataframes conformados caso partições ainda não tenham sido processadas, garantindo integridade estrita de schema e execução idempotente.

In [ ]:
import importlib.util
import inspect
import os
import shutil
import subprocess
import sys
from pathlib import Path

_REQUIRED_PACKAGES = (
    ("kagglehub", "kagglehub>=0.3.5"),
    ("polars", "polars"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
)
if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    for _mod_name, _pip_name in _REQUIRED_PACKAGES:
        if importlib.util.find_spec(_mod_name) is None:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "-U", _pip_name]
            )
    import kagglehub as _kh_check

    if "output_dir" not in inspect.signature(_kh_check.dataset_download).parameters:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=0.3.5"]
        )

# Contrato único de estado e ciclo de vida via KaggleHub SDK (Kaggle e Google Colab)
DATASET_HANDLE = os.getenv("KAGGLE_DATASET_HANDLE", "miolodados/tse-eleicoes-dados")
DATASET_PATH = os.getenv("DATASET_PATH", "/data/export/tse-eleicoes-dados")
STAGING_PATH = os.getenv("STAGING_PATH", "/tmp/tse-eleicoes-staging")
ALLOW_COLD_START = False


def is_directory_empty(path: Path) -> bool:
    """Verifica se DATASET_PATH está efetivamente vazio (ignorando arquivos ocultos de sistema)."""
    if not path.exists():
        return True
    return not any(item for item in path.iterdir() if not item.name.startswith("."))


def bootstrap_dataset(
    handle: str, target_dir: Path, allow_cold_start: bool = False
) -> Path:
    """Se DATASET_PATH estiver vazio, baixa a última versão do dataset do Kaggle e popula target_dir."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if not is_directory_empty(target_dir):
        print(f"[OK] DATASET_PATH ({target_dir}) já contém dados locais. Pulando download.")
        return target_dir

    print(
        f"[BOOTSTRAP] DATASET_PATH ({target_dir}) está vazio. Baixando '{handle}' via Kaggle SDK..."
    )
    try:
        import kagglehub

        try:
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    output_dir=str(target_dir),
                    force_download=True,
                )
            )
        except TypeError as type_err:
            if "output_dir" not in str(type_err):
                raise
            downloaded_path = Path(
                kagglehub.dataset_download(
                    handle,
                    force_download=True,
                )
            )

        if downloaded_path.resolve() != target_dir.resolve() and downloaded_path.exists():
            print(
                f"[SYNC] Copiando arquivos de {downloaded_path} (read-only mount) para {target_dir}..."
            )
            shutil.copytree(downloaded_path, target_dir, dirs_exist_ok=True)

        print(f"[OK] Dataset populado com sucesso em {target_dir}.")
    except Exception as exc:
        if allow_cold_start:
            print(
                f"[WARN] Dataset '{handle}' ainda não encontrado ou inacessível ({exc}). "
                f"Iniciando em modo Cold Start com {target_dir} vazio."
            )
        else:
            raise RuntimeError(
                f"Falha ao popular DATASET_PATH ({target_dir}) a partir do Kaggle ('{handle}'). "
                "Verifique sua autenticação e se as fases anteriores já publicaram o dataset."
            ) from exc

    return target_dir


if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    import kagglehub

    Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)
    Path(STAGING_PATH).mkdir(parents=True, exist_ok=True)
    print("Iniciando autenticação manual no Kaggle SDK...")
    kagglehub.login()
    bootstrap_dataset(
        DATASET_HANDLE, Path(DATASET_PATH), allow_cold_start=ALLOW_COLD_START
    )

import json
from datetime import UTC, datetime

import duckdb
import polars as pl
import pyarrow as pa

# ---------------------------------------------------------------------------
# Configuração de caminhos e parâmetros globais
# ---------------------------------------------------------------------------
ELECTION_YEAR = 2026
TURNS = [1, 2]
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ",
]

SILVER_BASE = Path(DATASET_PATH) / "silver"
BRONZE_BASE = Path(DATASET_PATH)
GOLD_OUT = Path(DATASET_PATH) / "gold"


def find_table_dir(table_name, roots=(Path(DATASET_PATH),), max_depth=7):
    """Localiza <table_name>/ com partições election_year=* sob DATASET_PATH."""
    for root in roots:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base >= max_depth:
                dirnames[:] = []
                continue
            if p.name == table_name and any(d.startswith("election_year=") for d in dirnames):
                return p
    return None


GOLD_OUT.mkdir(parents=True, exist_ok=True)

print("Ambiente configurado com sucesso:")
print(f"  Polars: {pl.__version__}")
print(f"  DuckDB: {duckdb.__version__}")
print(f"  PyArrow: {pa.__version__}")
print(f"  SILVER_BASE: {SILVER_BASE}")
print(f"  BRONZE_BASE: {BRONZE_BASE}")
print(f"  GOLD_OUT:    {GOLD_OUT}")

In [ ]:
# ---------------------------------------------------------------------------
# Schemas Silver canônicos para fallback defensivo
# ---------------------------------------------------------------------------
FACT_URNA_EVENT_SCHEMA = pl.Schema({
    "election_year": pl.Int16,
    "turn": pl.Int8,
    "uf": pl.String,
    "municipio_code": pl.Int32,
    "zone_num": pl.Int16,
    "section_num": pl.Int32,
    "urna_serial": pl.String,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Datetime("us"),
    "has_timestamp_regression": pl.Boolean,
    "application": pl.String,
    "log_level": pl.String,
    "event_code": pl.String,
    "biometric_attempt_num": pl.Int8,
    "classification_status": pl.String,
    "parser_version": pl.String,
})

FACT_VOTING_SESSION_SCHEMA = pl.Schema({
    "election_year": pl.Int16,
    "turn": pl.Int8,
    "uf": pl.String,
    "municipio_code": pl.Int32,
    "zone_num": pl.Int16,
    "section_num": pl.Int32,
    "urna_serial": pl.String,
    "first_event_ts": pl.Datetime("us"),
    "last_event_ts": pl.Datetime("us"),
    "final_state": pl.String,
    "is_complete": pl.Boolean,
    "has_contingency": pl.Boolean,
    "voter_cycles": pl.Int32,
    "event_count": pl.Int32,
    "state_machine_version": pl.String,
})

def scan_silver_table(table_name: str, schema: pl.Schema) -> tuple[pl.DataFrame, list[str]]:
    """Busca defensivamente arquivos Parquet de tabelas Silver em diretórios candidatos.
    
    Se nenhum arquivo for encontrado ou a leitura retornar 0 registros,
    retorna um DataFrame vazio estritamente tipado conforme o schema canônico.
    """
    candidate_dirs = [
        find_table_dir(table_name) or SILVER_BASE / table_name,
        SILVER_BASE / table_name,
        Path(DATASET_PATH) / "silver" / table_name,
        Path(DATASET_PATH) / table_name,
    ]
    
    found_files: list[Path] = []
    for cand in candidate_dirs:
        if cand.exists():
            files = list(cand.glob("**/*.parquet"))
            if files:
                found_files.extend(files)
                print(f"[{table_name}] Encontrados {len(files)} arquivos em {cand}")
    
    unique_files = list(dict.fromkeys(found_files))
    
    if not unique_files:
        raise FileNotFoundError(f"[{table_name}] Nenhum Parquet Silver localizado. Execute os NB 05/06 e anexe suas saídas (kernel_sources).")
    
    try:
        df = pl.read_parquet(unique_files)
        select_exprs = []
        for col_name, col_type in schema.items():
            if col_name in df.columns:
                select_exprs.append(pl.col(col_name).cast(col_type))
            else:
                select_exprs.append(pl.lit(None).cast(col_type).alias(col_name))
        df_conformed = df.select(select_exprs)
        print(f"[{table_name}] Carregados {df_conformed.height} registros a partir de {len(unique_files)} arquivos.")
        return df_conformed, [str(f) for f in unique_files]
    except Exception as exc:
        print(f"[{table_name}] Aviso na leitura dos arquivos ({exc}). Fallback para DataFrame vazio com schema canônico.")
        return pl.DataFrame(schema=schema), [str(f) for f in unique_files]

df_events, event_files = scan_silver_table("fact_urna_event", FACT_URNA_EVENT_SCHEMA)
df_sessions, session_files = scan_silver_table("fact_voting_session", FACT_VOTING_SESSION_SCHEMA)

print("\nResumo de Carga de Entrada:")
print(f"  fact_urna_event:      {df_events.height} linhas ({len(event_files)} arquivos)")
print(f"  fact_voting_session:  {df_sessions.height} linhas ({len(session_files)} arquivos)")
if df_events.height == 0 or df_sessions.height == 0:
    raise RuntimeError("Silver vazia: Gold não pode ser gerada a partir de tabelas sem linhas.")

In [ ]:
# ---------------------------------------------------------------------------
# Produto Gold 1: gold_event_distribution
# Grão: (event_code, turn, uf)
# Métrica: contagem de eventos e proporção percentual relativa à UF e Turno
# ---------------------------------------------------------------------------
GOLD_EVENT_DIST_SCHEMA = pl.Schema({
    "event_code": pl.String,
    "turn": pl.Int8,
    "uf": pl.String,
    "event_count": pl.Int64,
    "total_uf_events": pl.Int64,
    "percentage": pl.Float64,
})

if df_events.is_empty():
    print("[gold_event_distribution] Entrada vazia; gerando DataFrame vazio conforme schema.")
    gold_event_dist = pl.DataFrame(schema=GOLD_EVENT_DIST_SCHEMA)
else:
    gold_event_dist = (
        df_events
        .group_by(["event_code", "turn", "uf"])
        .agg(pl.len().cast(pl.Int64).alias("event_count"))
        .with_columns(
            pl.col("event_count").sum().over(["turn", "uf"]).alias("total_uf_events")
        )
        .with_columns(
            pl.when(pl.col("total_uf_events") > 0)
              .then((pl.col("event_count") * 100.0 / pl.col("total_uf_events")).round(4))
              .otherwise(0.0)
              .alias("percentage")
        )
        .sort(["turn", "uf", "event_count"], descending=[False, False, True])
        .select(["event_code", "turn", "uf", "event_count", "total_uf_events", "percentage"])
    )

event_dist_path = GOLD_OUT / "gold_event_distribution.parquet"
gold_event_dist.write_parquet(event_dist_path, compression="zstd", compression_level=5)

print(f"[gold_event_distribution] Salvo em: {event_dist_path}")
print(f"  Linhas materializadas: {gold_event_dist.height}")
print(f"  Schema: {gold_event_dist.schema}")
if gold_event_dist.height > 0:
    print("\nAmostra (primeiras 5 linhas):")
    print(gold_event_dist.head(5))

In [ ]:
# ---------------------------------------------------------------------------
# Produto Gold 2: gold_session_completeness
# Grão: (turn, uf, final_state)
# Métricas: contagem de sessões, taxa de conclusão, total e média de ciclos de votantes
# ---------------------------------------------------------------------------
GOLD_SESSION_COMP_SCHEMA = pl.Schema({
    "turn": pl.Int8,
    "uf": pl.String,
    "final_state": pl.String,
    "session_count": pl.Int64,
    "total_sessions_uf": pl.Int64,
    "state_share_pct": pl.Float64,
    "completion_rate": pl.Float64,
    "total_voter_cycles": pl.Int64,
    "avg_voter_cycles": pl.Float64,
    "min_voter_cycles": pl.Int32,
    "max_voter_cycles": pl.Int32,
    "total_events": pl.Int64,
})

if df_sessions.is_empty():
    print("[gold_session_completeness] Entrada vazia; gerando DataFrame vazio conforme schema.")
    gold_session_comp = pl.DataFrame(schema=GOLD_SESSION_COMP_SCHEMA)
else:
    gold_session_comp = (
        df_sessions
        .group_by(["turn", "uf", "final_state"])
        .agg([
            pl.len().cast(pl.Int64).alias("session_count"),
            pl.col("voter_cycles").sum().cast(pl.Int64).alias("total_voter_cycles"),
            pl.col("voter_cycles").mean().round(2).alias("avg_voter_cycles"),
            pl.col("voter_cycles").min().cast(pl.Int32).alias("min_voter_cycles"),
            pl.col("voter_cycles").max().cast(pl.Int32).alias("max_voter_cycles"),
            pl.col("event_count").sum().cast(pl.Int64).alias("total_events"),
        ])
        .with_columns(
            pl.col("session_count").sum().over(["turn", "uf"]).alias("total_sessions_uf"),
            pl.when(pl.col("final_state") == "CLOSED")
              .then(pl.col("session_count"))
              .otherwise(0)
              .sum()
              .over(["turn", "uf"])
              .alias("_closed_count_uf")
        )
        .with_columns([
            pl.when(pl.col("total_sessions_uf") > 0)
              .then((pl.col("session_count") * 100.0 / pl.col("total_sessions_uf")).round(4))
              .otherwise(0.0)
              .alias("state_share_pct"),
            pl.when(pl.col("total_sessions_uf") > 0)
              .then((pl.col("_closed_count_uf") * 100.0 / pl.col("total_sessions_uf")).round(4))
              .otherwise(0.0)
              .alias("completion_rate"),
        ])
        .drop("_closed_count_uf")
        .sort(["turn", "uf", "final_state"])
        .select([
            "turn", "uf", "final_state", "session_count", "total_sessions_uf",
            "state_share_pct", "completion_rate", "total_voter_cycles",
            "avg_voter_cycles", "min_voter_cycles", "max_voter_cycles", "total_events"
        ])
    )

session_comp_path = GOLD_OUT / "gold_session_completeness.parquet"
gold_session_comp.write_parquet(session_comp_path, compression="zstd", compression_level=5)

print(f"[gold_session_completeness] Salvo em: {session_comp_path}")
print(f"  Linhas materializadas: {gold_session_comp.height}")
print(f"  Schema: {gold_session_comp.schema}")
if gold_session_comp.height > 0:
    print("\nAmostra (primeiras 5 linhas):")
    print(gold_session_comp.head(5))

In [ ]:
# ---------------------------------------------------------------------------
# Produto Gold 3: gold_biometric_attempts
# Grão: (turn, uf, biometric_attempt_num)
# Métricas: contagem de tentativas e percentual cumulativo por UF e Turno
# ---------------------------------------------------------------------------
GOLD_BIO_ATTEMPTS_SCHEMA = pl.Schema({
    "turn": pl.Int8,
    "uf": pl.String,
    "biometric_attempt_num": pl.Int8,
    "attempt_count": pl.Int64,
    "total_attempts_uf": pl.Int64,
    "percentage": pl.Float64,
    "cumulative_count": pl.Int64,
    "cumulative_share": pl.Float64,
})

df_bio = df_events.filter(
    pl.col("biometric_attempt_num").is_not_null() & (pl.col("biometric_attempt_num") > 0)
)

if df_bio.is_empty():
    print("[gold_biometric_attempts] Nenhuma tentativa biométrica identificada; gerando DataFrame vazio conforme schema.")
    gold_bio_attempts = pl.DataFrame(schema=GOLD_BIO_ATTEMPTS_SCHEMA)
else:
    gold_bio_attempts = (
        df_bio
        .group_by(["turn", "uf", "biometric_attempt_num"])
        .agg(pl.len().cast(pl.Int64).alias("attempt_count"))
        .sort(["turn", "uf", "biometric_attempt_num"])
        .with_columns(
            pl.col("attempt_count").sum().over(["turn", "uf"]).alias("total_attempts_uf"),
            pl.col("attempt_count").cum_sum().over(["turn", "uf"]).alias("cumulative_count"),
        )
        .with_columns([
            pl.when(pl.col("total_attempts_uf") > 0)
              .then((pl.col("attempt_count") * 100.0 / pl.col("total_attempts_uf")).round(4))
              .otherwise(0.0)
              .alias("percentage"),
            pl.when(pl.col("total_attempts_uf") > 0)
              .then((pl.col("cumulative_count") * 100.0 / pl.col("total_attempts_uf")).round(4))
              .otherwise(0.0)
              .alias("cumulative_share"),
        ])
        .select([
            "turn", "uf", "biometric_attempt_num", "attempt_count",
            "total_attempts_uf", "percentage", "cumulative_count", "cumulative_share"
        ])
    )

bio_attempts_path = GOLD_OUT / "gold_biometric_attempts.parquet"
gold_bio_attempts.write_parquet(bio_attempts_path, compression="zstd", compression_level=5)

print(f"[gold_biometric_attempts] Salvo em: {bio_attempts_path}")
print(f"  Linhas materializadas: {gold_bio_attempts.height}")
print(f"  Schema: {gold_bio_attempts.schema}")
if gold_bio_attempts.height > 0:
    print("\nAmostra (primeiras 5 linhas):")
    print(gold_bio_attempts.head(5))

In [ ]:
# ---------------------------------------------------------------------------
# Validações, Asserções e Reconciliação com DuckDB
# ---------------------------------------------------------------------------
print("Iniciando bateria de asserções e testes de consistência...")

# 1. Validação de gold_event_distribution
if df_events.height > 0:
    reconciled_events = gold_event_dist["event_count"].sum()
    assert reconciled_events == df_events.height, (
        f"Mismatched event count: sum in gold={reconciled_events} vs input={df_events.height}"
    )
    null_dims = gold_event_dist.filter(
        pl.col("event_code").is_null() | pl.col("turn").is_null() | pl.col("uf").is_null()
    ).height
    assert null_dims == 0, f"Encontrados {null_dims} nulos em dimensões de gold_event_distribution"
    invalid_pcts = gold_event_dist.filter(
        (pl.col("percentage") < 0.0) | (pl.col("percentage") > 100.0)
    ).height
    assert invalid_pcts == 0, f"Encontradas porcentagens inválidas em gold_event_distribution: {invalid_pcts}"
else:
    assert gold_event_dist.height == 0, "gold_event_distribution deveria ter 0 linhas para entrada vazia"
print("✓ gold_event_distribution: reconciliação e dimensões válidas.")

# 2. Validação de gold_session_completeness
if df_sessions.height > 0:
    reconciled_sessions = gold_session_comp["session_count"].sum()
    assert reconciled_sessions == df_sessions.height, (
        f"Mismatched session count: sum in gold={reconciled_sessions} vs input={df_sessions.height}"
    )
    null_session_dims = gold_session_comp.filter(
        pl.col("turn").is_null() | pl.col("uf").is_null() | pl.col("final_state").is_null()
    ).height
    assert null_session_dims == 0, f"Encontrados {null_session_dims} nulos em dimensões de gold_session_completeness"
    invalid_rates = gold_session_comp.filter(
        (pl.col("completion_rate") < 0.0) | (pl.col("completion_rate") > 100.0)
    ).height
    assert invalid_rates == 0, f"Taxas de completude fora do intervalo [0, 100]: {invalid_rates}"
else:
    assert gold_session_comp.height == 0, "gold_session_completeness deveria ter 0 linhas para entrada vazia"
print("✓ gold_session_completeness: reconciliação e integridade de estado válidas.")

# 3. Validação de gold_biometric_attempts
if df_bio.height > 0:
    reconciled_bio = gold_bio_attempts["attempt_count"].sum()
    assert reconciled_bio == df_bio.height, (
        f"Mismatched biometric count: sum in gold={reconciled_bio} vs input={df_bio.height}"
    )
    null_bio_dims = gold_bio_attempts.filter(
        pl.col("turn").is_null() | pl.col("uf").is_null() | pl.col("biometric_attempt_num").is_null()
    ).height
    assert null_bio_dims == 0, f"Encontrados {null_bio_dims} nulos em dimensões de gold_biometric_attempts"
    invalid_shares = gold_bio_attempts.filter(
        (pl.col("cumulative_share") < 0.0) | (pl.col("cumulative_share") > 100.001)
    ).height
    assert invalid_shares == 0, f"Cumulative share fora do intervalo [0, 100]: {invalid_shares}"
else:
    assert gold_bio_attempts.height == 0, "gold_biometric_attempts deveria ter 0 linhas para entrada vazia"
print("✓ gold_biometric_attempts: reconciliação e curva cumulativa válidas.")

# 4. Verificação independente de Parquets via DuckDB
con = duckdb.connect()
tables = [
    ("gold_event_distribution", event_dist_path),
    ("gold_session_completeness", session_comp_path),
    ("gold_biometric_attempts", bio_attempts_path),
]
for tname, tpath in tables:
    duck_count = con.execute(f"SELECT count(*) FROM read_parquet('{tpath}')").fetchone()[0]
    print(f"✓ DuckDB verificou {tname}: {duck_count} registros lidos de {tpath.name}")
con.close()

print("\nTodas as asserções e checagens de consistência passaram com sucesso!")

In [ ]:
# ---------------------------------------------------------------------------
# Geração e Publicação do Manifesto Gold
# ---------------------------------------------------------------------------
manifest_data = {
    "generated_at": datetime.now(UTC).isoformat(),
    "pipeline_stage": "gold",
    "version": "1.0.0",
    "tables": {
        "gold_event_distribution": {
            "path": str(event_dist_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_event_dist.height,
            "grain": "(event_code, turn, uf)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_event_dist.schema.items()],
        },
        "gold_session_completeness": {
            "path": str(session_comp_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_session_comp.height,
            "grain": "(turn, uf, final_state)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_session_comp.schema.items()],
        },
        "gold_biometric_attempts": {
            "path": str(bio_attempts_path.relative_to(GOLD_OUT.parent)),
            "rows": gold_bio_attempts.height,
            "grain": "(turn, uf, biometric_attempt_num)",
            "columns": [{"name": c, "type": str(t)} for c, t in gold_bio_attempts.schema.items()],
        },
    },
    "inputs_scanned": {
        "fact_urna_event": {
            "rows": df_events.height,
            "files_found": len(event_files),
        },
        "fact_voting_session": {
            "rows": df_sessions.height,
            "files_found": len(session_files),
        },
    },
    "status": "SUCCESS",
}

manifest_file = GOLD_OUT / "manifest.json"
with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest_data, f, indent=2, ensure_ascii=False)

print(f"Manifesto Gold gravado em: {manifest_file}\n")
print(json.dumps(manifest_data, indent=2, ensure_ascii=False))

# Validação de higiene pré-upload e publicação de nova versão via KaggleHub SDK
PHASE_ID = "08_gold_analytics"
PHASE_SUMMARY = "Camada Gold: produtos analíticos (distribuição de eventos, completude de sessões, biometria)"

forbidden_extensions = {".zip", ".jez", ".tmp", ".part"}
offending_files = [
    str(p.relative_to(DATASET_PATH))
    for p in Path(DATASET_PATH).rglob("*")
    if p.is_file() and p.suffix.lower() in forbidden_extensions
]
if offending_files:
    raise ValueError(
        f"Upload abortado! Arquivos brutos/temporários detectados em DATASET_PATH: {offending_files[:10]}"
    )

if is_directory_empty(Path(DATASET_PATH)):
    raise RuntimeError(
        f"Upload abortado! DATASET_PATH ({DATASET_PATH}) está vazio após o processamento."
    )

if globals().get("__name__") == "__main__" and not os.getenv("PYTEST_CURRENT_TEST"):
    import kagglehub

    print(f"Publicando nova versão do dataset '{DATASET_HANDLE}' a partir de {DATASET_PATH}...")
    kagglehub.dataset_upload(
        handle=DATASET_HANDLE,
        local_dataset_dir=str(DATASET_PATH),
        version_notes=f"[{PHASE_ID}] {PHASE_SUMMARY}",
        ignore_patterns=[
            "*.tmp",
            "*.part",
            "*.zip",
            "*.jez",
            "*.log",
            "__pycache__/",
            ".ipynb_checkpoints/",
            ".DS_Store",
        ],
    )
    print(f"[SUCESSO] Nova versão de '{DATASET_HANDLE}' publicada no Kaggle!")

## Limitações e Definição das Métricas

### 1. Definição das Métricas e Fórmulas

| Tabela Gold | Grão | Métricas Chave | Fórmula / Regra | Propósito Analítico |
|---|---|---|---|---|
| `gold_event_distribution` | `(event_code, turn, uf)` | `event_count`, `total_uf_events`, `percentage` | `percentage = (event_count / total_uf_events) * 100.0` | Medir a prevalência e frequência de cada categoria de evento da urna por estado e turno. |
| `gold_session_completeness` | `(turn, uf, final_state)` | `session_count`, `state_share_pct`, `completion_rate`, `total_voter_cycles`, `avg_voter_cycles` | `completion_rate = (closed_sessions / total_sessions) * 100.0`; `voter_cycles = sum(VOTER_VOTE_COMPUTED)` | Acompanhar taxa de encerramento normal vs seções incompletas e o volume de votos computados. |
| `gold_biometric_attempts` | `(turn, uf, biometric_attempt_num)` | `attempt_count`, `total_attempts_uf`, `percentage`, `cumulative_share` | `cumulative_share = cum_sum(attempt_count) / total_attempts * 100.0` | Avaliar distribuição de tentativas biométricas e taxa de identificação por tentativa sucessiva. |

### 2. Filtros e Regras Aplicadas
- **Eventos:** Não há filtros excludentes; eventos categorizados e desconhecidos (`UNKNOWN_EVENT`) são contabilizados para total fidelidade aos logs processados.
- **Biometria:** Filtrados estritamente registros onde `biometric_attempt_num > 0`. Tentativas manuais de liberação de voto por mesário possuem eventos específicos no catálogo e não são computadas como tentativa biométrica ordinal.
- **Sessões:** Agrupadas a partir de `fact_voting_session` gerada pela máquina de estados determinística v1.0.0.

### 3. Limitações e Ressalvas
- **Janela Observada:** As sessões reconstruídas refletem os eventos presentes nos arquivos de log disponíveis. Urnas onde arquivos parciais foram recuperados podem indicar `INCOMPLETE` mesmo tendo sido encerradas normalmente após o encerramento do log.
- **Execução Desacoplada e Idempotente:** Caso as partições de entrada Silver ainda não tenham sido geradas no ambiente de execução, o pipeline produz tabelas vazias com esquemas conformados e manifesto válido, preservando contratos de schema downstream sem interrupções.